# 02 — Sliding Window Strategy — Practical Examples

**Covers**: Keep-N, token-aware window, paired turns, pinned+sliding, ContextWindow class, agent loop integration

In [ ]:
import os, json
import tiktoken
from openai import OpenAI
from dotenv import load_dotenv
from rich import print as rprint

load_dotenv()
client = OpenAI()
MODEL = 'gpt-4o-mini'
enc = tiktoken.encoding_for_model(MODEL)

def count_tokens(messages: list[dict]) -> int:
    total = 3
    for m in messages:
        total += 3 + len(enc.encode(str(m.get('content', ''))))
    return total

print('✅ Setup complete')

## 1. Keep-N — Simplest Window

In [ ]:
def keep_n_messages(messages: list[dict], n: int, always_keep_system: bool = True) -> list[dict]:
    if not messages: return messages
    if always_keep_system and messages[0]['role'] == 'system':
        return [messages[0]] + messages[1:][-n:]
    return messages[-n:]

# Build a 15-message history
msgs = [{"role": "system", "content": "You are a helpful coding assistant."}]
for i in range(1, 8):
    msgs.append({"role": "user",      "content": f"User question {i}: explain concept {i}"})
    msgs.append({"role": "assistant", "content": f"Assistant answer {i}: concept {i} means X, Y, Z with important details."})

print(f"Original: {len(msgs)} messages, {count_tokens(msgs):,} tokens")

for n in [4, 6, 10]:
    trimmed = keep_n_messages(msgs, n)
    print(f"Keep-{n}:  {len(trimmed)} messages, {count_tokens(trimmed):,} tokens | Lost {len(msgs)-len(trimmed)} messages")

## 2. Token-Aware Sliding Window

In [ ]:
def sliding_window_tokens(messages: list[dict], max_tokens: int = 1500) -> list[dict]:
    """Keep newest messages within token budget. System message always preserved."""
    if not messages: return messages
    
    def count_msg(m): return 3 + len(enc.encode(str(m.get('content', ''))))
    
    if messages[0]['role'] == 'system':
        system_msg = messages[0]
        history = messages[1:]
        used = count_msg(system_msg) + 3
    else:
        system_msg = None
        history = messages
        used = 3
    
    kept = []
    for msg in reversed(history):
        t = count_msg(msg)
        if used + t <= max_tokens:
            kept.append(msg)
            used += t
        else:
            break
    kept.reverse()
    
    return ([system_msg] if system_msg else []) + kept

# Compare budgets
for budget in [500, 1000, 2000]:
    trimmed = sliding_window_tokens(msgs, budget)
    actual = count_tokens(trimmed)
    dropped = len(msgs) - len(trimmed)
    print(f"Budget {budget:,}: {len(trimmed)} msgs, {actual:,} tokens used | dropped {dropped} messages")

## 3. Paired Turns — Keep Whole User+Assistant Pairs

In [ ]:
def keep_n_turns(messages: list[dict], n_turns: int) -> list[dict]:
    """Keep last N complete user+assistant pairs."""
    if not messages: return messages
    system = [messages[0]] if messages[0]['role'] == 'system' else []
    history = messages[len(system):]
    
    turns = []
    i = 0
    while i < len(history) - 1:
        if history[i]['role'] == 'user' and history[i+1]['role'] == 'assistant':
            turns.append((history[i], history[i+1]))
            i += 2
        else:
            i += 1
    
    recent = turns[-n_turns:]
    flat = [m for pair in recent for m in pair]
    return system + flat

for n in [2, 3, 5]:
    trimmed = keep_n_turns(msgs, n)
    print(f"Keep {n} turns: {len(trimmed)} messages, {count_tokens(trimmed):,} tokens")

## 4. ContextWindow Class — Production Implementation

In [ ]:
class ContextWindow:
    """Token-aware sliding context window. System prompt always pinned."""
    
    def __init__(self, system: str, max_history_tokens: int = 2000, model: str = MODEL):
        self.system = system
        self.max_history_tokens = max_history_tokens
        self.enc = tiktoken.encoding_for_model(model)
        self._history: list[dict] = []
        self._dropped_count = 0
    
    def add(self, role: str, content: str):
        self._history.append({'role': role, 'content': content})
    
    def _count_msg(self, m: dict) -> int:
        return 3 + len(self.enc.encode(str(m.get('content', ''))))
    
    def _trim(self) -> list[dict]:
        kept, used = [], 0
        for msg in reversed(self._history):
            t = self._count_msg(msg)
            if used + t <= self.max_history_tokens:
                kept.append(msg)
                used += t
            else:
                self._dropped_count += 1
        kept.reverse()
        return kept
    
    def get_messages(self) -> list[dict]:
        return [{'role': 'system', 'content': self.system}] + self._trim()
    
    def stats(self) -> dict:
        msgs = self.get_messages()
        tokens = sum(self._count_msg(m) for m in msgs) + 3
        return {'total_history': len(self._history), 'window_messages': len(msgs),
                'window_tokens': tokens, 'dropped_total': self._dropped_count}

# Use in a real agent loop
cw = ContextWindow(
    system="You are a helpful Python coding assistant.",
    max_history_tokens=800
)

questions = [
    "What is a list comprehension?",
    "How do I use enumerate()?",
    "What are *args and **kwargs?",
    "Explain decorators.",
    "What is a context manager?"
]

for q in questions:
    cw.add('user', q)
    msgs = cw.get_messages()
    r = client.chat.completions.create(model=MODEL, messages=msgs, max_tokens=120)
    answer = r.choices[0].message.content
    cw.add('assistant', answer[:200])  # Truncate long answers for demo
    stats = cw.stats()
    print(f"Q: {q[:40]:<42} | Window: {stats['window_messages']} msgs, {stats['window_tokens']:,} tokens")

print(f"\nFinal stats: {cw.stats()}")

## 5. Visualizing What Gets Dropped

In [ ]:
# Make context management visible — what's in vs what's dropped
def show_window(original: list[dict], windowed: list[dict]):
    orig_roles = [(i, m['role']) for i, m in enumerate(original)]
    wind_set   = set(id(m) for m in windowed)
    
    print(f"{'#':<4} {'Role':<12} {'Status':<10} {'Content Preview'}")
    print('-' * 65)
    for i, m in enumerate(original):
        status = '✅ IN' if id(m) in wind_set else '❌ OUT'
        preview = str(m.get('content', ''))[:40]
        print(f"{i:<4} {m['role']:<12} {status:<10} {preview}")

# Build & trim
test_msgs = [
    {'role': 'system',    'content': 'You are helpful.'},
    {'role': 'user',      'content': 'Tell me about Python'},
    {'role': 'assistant', 'content': 'Python is a versatile programming language known for readability.'},
    {'role': 'user',      'content': 'What are its main uses?'},
    {'role': 'assistant', 'content': 'Python is used in web development, data science, AI/ML, automation, and scripting.'},
    {'role': 'user',      'content': 'Which frameworks are popular?'},
    {'role': 'assistant', 'content': 'Django, FastAPI, Flask for web. TensorFlow, PyTorch for ML. Pandas, NumPy for data.'},
]

trimmed = sliding_window_tokens(test_msgs, max_tokens=300)
print(f"Original: {len(test_msgs)} messages | Trimmed: {len(trimmed)} messages\n")
show_window(test_msgs, trimmed)

## 📌 Summary

- **Keep-N**: fastest — trim to last N messages (with system pinned)
- **Token-aware**: precise — trim newest-to-oldest until budget fits
- **Paired turns**: keeps user+assistant pairs clean — no orphan messages
- **ContextWindow class**: production-ready — pin system, auto-trim, track stats
- **Always pin the system prompt** — it's the agent's operating manual
- **Start trimming at 50-60% of limit** — not right at the edge